In [1]:
# %%
# pip install matplotlib xgboost shap seaborn scikit-learn
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV

# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Feature Importance
import shap

# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os


c:\Users\Phuc\OneDrive\Tài liệu\grad_thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# FUNCTIONS

In [2]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    m = y_true != 0
    return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100

def evaluate_model(name, model):
    return {
        'Model':      name,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, model.predict(X_train))),
        'Test RMSE':  np.sqrt(mean_squared_error(y_test,  model.predict(X_test))),
        'Train MAE':  mean_absolute_error(y_train, model.predict(X_train)),
        'Test MAE':   mean_absolute_error(y_test,  model.predict(X_test)),
        'Train MAPE': mape(y_train, model.predict(X_train)),
        'Test MAPE':  mape(y_test,  model.predict(X_test)),
        'Train R2':   r2_score(y_train, model.predict(X_train)),
        'Test R2':    r2_score(y_test,  model.predict(X_test)),
    }

# LOAD ALL FEATURE TABLES

In [10]:
print("Loading feature tables...")

base_dir = r'..\data\features'

nps   = pd.read_csv(os.path.join(base_dir, 'nps_aggregated_features.csv'))
txn   = pd.read_csv(os.path.join(base_dir, 'transaction_features.csv'))
deliv = pd.read_excel(os.path.join(base_dir, 'delivery_features.xlsx'))
plano = pd.read_csv(os.path.join(base_dir, 'planogram_features.csv'))
shop  = pd.read_csv(os.path.join(base_dir, 'shop.csv')) 
ms    = pd.read_excel(os.path.join(base_dir, 'ms_features.xlsx')) # Sửa .xslx thành .xlsx

# Convert opening date to datetime if not already
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')
deliv = deliv.rename(columns={
    'cua_hang': 'store_id'
})
print(f"  NPS: {nps.shape} | TXN: {txn.shape} | DELIV: {deliv.shape} | PLANO: {plano.shape} | SHOP: {shop.shape} | MS: {ms.shape}")


Loading feature tables...
  NPS: (627, 10) | TXN: (963, 8) | DELIV: (942, 9) | PLANO: (757, 9) | SHOP: (164, 10) | MS: (938, 7)


# MERGE & TENURE

In [17]:
# Merge datasets starting from a base combination (txn and nps)
merged = pd.merge(txn, nps, on=['store_id', 'YearMonth'], how='inner')
merged = pd.merge(merged, deliv, on=['store_id', 'YearMonth'], how='left')  
merged = pd.merge(merged, plano, on=['store_id', 'YearMonth'], how='left')

# Filter to Oct 2025 – Feb 2026
merged = merged[merged['YearMonth'].between('2025-10', '2026-02')].copy()

# Merge with shop_info
merged = pd.merge(merged, shop, on='store_id', how='left')

# ── Create tenure column ──
merged['_period'] = pd.to_datetime(merged['YearMonth'], format='%Y-%m')
merged['tenure'] = (
    (merged['_period'].dt.year  - merged['khai_truong_date'].dt.year)  * 12 +
    (merged['_period'].dt.month - merged['khai_truong_date'].dt.month)
)
merged.drop(columns=['_period', 'khai_truong_date'], inplace=True)

print(f"\nFinal merged dataset: {merged.shape}")
print(f"Months: {sorted(merged['YearMonth'].unique())}")


Final merged dataset: (479, 39)
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02']


# FEATURE / TARGET SPLIT & PEARSON CORRELATION

In [19]:
TARGET    = 'nps_weighted_score'

# Exclude identification columns and calculation intermediates from features
DROP_COLS = [
    'store_id', 'YearMonth', TARGET, 
    'satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket'
]
feature_cols = [c for c in merged.columns if c not in DROP_COLS]

X = merged[feature_cols]
y = merged[TARGET]

# Drop rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f"\nRows after NaN drop: {len(X)}")

# Compute Pearson Correlation
print("\nCalculating Pearson Correlation Matrix...")
corr_matrix = X.corr(method='pearson')

# Plot and save correlation matrix
plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix")
plt.savefig(os.path.join(base_dir, 'pearson_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()

# Identify features with high collinearity (>0.7)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column].abs() > 0.7)]
print(f"Features to drop due to multicollinearity (|corr| > 0.7): {to_drop}")

# Drop identified variables
X = X.drop(columns=to_drop)
feature_cols = X.columns.tolist()
print(f"Features after dropping multicollinear terms ({len(feature_cols)}): {feature_cols}")



Rows after NaN drop: 343

Calculating Pearson Correlation Matrix...
Features to drop due to multicollinearity (|corr| > 0.7): ['num_product', 'order_delivered', 'order_pickup', 'order_onsite', 'speed_median_hours', 'late_rate', 'err_basic', 'err_display', 'err_posm', 'sizeshop']
Features after dropping multicollinear terms (23): ['num_order', 'pct_delivered', 'order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets', 'speed_mean_hours', 'delay_mean_hours', 'delay_max_hours', 'promise_gap_mean_hours', 'ontime_rate', 'planogram_score', 'err_electronic_price_board', 'err_operations', 'err_advisory_partition', 'tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2', 'kho_m2', 'via_he_rongm', 'is_southern', 'tenure']


In [ ]:
X

# TRAIN / TEST SPLIT

In [ ]:
_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

# SECTION 6 — GRIDSEARCHCV MODEL TRAINING

In [ ]:
results     = []
best_models = {}

# ── Decision Tree ──
print("\n[1/3] Tuning Decision Tree...")
dt_gs = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
dt_gs.fit(X_train, y_train)
best_models['Decision Tree'] = dt_gs.best_estimator_
results.append(evaluate_model('Decision Tree', dt_gs.best_estimator_))
print(f"   Best params: {dt_gs.best_params_}")

# ── Random Forest ──
print("\n[2/3] Tuning Random Forest...")
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 200, 300], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
best_models['Random Forest'] = rf_gs.best_estimator_
results.append(evaluate_model('Random Forest', rf_gs.best_estimator_))
print(f"   Best params: {rf_gs.best_params_}")

# ── XGBoost ──
print("\n[3/3] Tuning XGBoost...")
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7, 1.0]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
best_models['XGBoost'] = xgb_gs.best_estimator_
results.append(evaluate_model('XGBoost', xgb_gs.best_estimator_))
print(f"   Best params: {xgb_gs.best_params_}")


# EVALUATION TABLE

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print("\n" + "="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(results_df[['Train MAE', 'Test MAE', 'Train MAPE', 'Test MAPE', 'Train R2', 'Test R2']].round(4))

results_df.to_csv(os.path.join(base_dir, 'model_results.csv'))
